<a href="https://colab.research.google.com/github/KravitzLab/Murrell2026/blob/main/Figures/Murrell_2026_Fig3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Murrell 2026 Figure 3

Authors: Chantelle Murrell<br>
Updated: 03/04/26

In [ ]:
# @title Install libraries and import them {"run":"auto"}

import importlib.util
import subprocess
import sys

# Packages to ensure are installed (add others here if you like)
packages = {
    "fed3": "git+https://github.com/earnestt1234/fed3.git",
    "fed3bandit": "fed3bandit",
    "pingouin": "pingouin",
    "ipydatagrid": "ipydatagrid",
    "openpyxl": "openpyxl",
}

for name, source in packages.items():
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", source])

# ----------------------------
# Imports
# ----------------------------
# Standard library
import copy
import io
import math
import os
import re
import shutil
import tempfile
import threading
import time
import warnings
import zipfile
import requests
import glob
from datetime import datetime, timedelta
from os.path import basename, splitext

# Third-party
from ipydatagrid import DataGrid, TextRenderer
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
import fed3
import fed3.plot as fplot
import fed3bandit as f3b
from scipy.stats import f_oneway, linregress, ttest_ind
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.gridspec as gridspec
from matplotlib.ticker import PercentFormatter
from google.colab import files
try:
    from tqdm.auto import tqdm   # nice in notebooks; falls back to std tqdm on console
except Exception:
    # safe no-op fallback if tqdm isn't installed
    def tqdm(x):
        return x


# ----------------------------
# Configuration
# ----------------------------
warnings.filterwarnings("ignore")
plt.rcParams.update({'font.size': 12, 'figure.autolayout': True})
plt.rcParams['figure.figsize'] = [6, 4]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


plt.rcParams.update({'font.size': 12})

RNG_GLOBAL = 123
np.random.seed(RNG_GLOBAL)

# --- WSLS simulation utilities ---
def simulate_mean_pHigh(
    win_stay,
    p_high,
    p_low,
    n_mice=100,
    total_pellets=300,
    pellets_per_block=20,
    lose_shift=0.5,
    lapse=0.25,
    seed=0,
):
    """Return (mean_pHigh, sem_pHigh) across n_mice."""
    rng = np.random.default_rng(seed)
    acc = []
    for _ in range(n_mice):
        rich = rng.integers(0, 2)  # 0=Left, 1=Right
        pellets_in_block = 0
        pellets_earned = 0
        prev_choice = None
        prev_reward = None
        correct = 0
        total_trials = 0

        while pellets_earned < total_pellets:
            if prev_choice is None:
                choice = rng.integers(0, 2)
            else:
                if rng.random() < lapse:
                    choice = rng.integers(0, 2)
                else:
                    if prev_reward == 1:
                        choice = prev_choice if rng.random() < win_stay else 1 - prev_choice
                    else:
                        choice = 1 - prev_choice if rng.random() < lose_shift else prev_choice

            rich_pre = rich
            reward = rng.random() < (p_high if choice == rich_pre else p_low)

            correct += int(choice == rich_pre)
            total_trials += 1

            if reward:
                pellets_earned += 1
                pellets_in_block += 1
                if pellets_in_block >= pellets_per_block:
                    rich = 1 - rich
                    pellets_in_block = 0

            prev_choice, prev_reward = choice, int(reward)

        acc.append(correct / total_trials)

    acc = np.asarray(acc, dtype=float)
    return float(acc.mean()), float(acc.std(ddof=1) / np.sqrt(len(acc)))

print("Packages installed and imports ready.")


In [ ]:
# @title Import FED3 Bandit 80-20 Data

from urllib.request import urlretrieve
import os, zipfile, shutil
import pandas as pd
import numpy as np

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

zip_url = "https://github.com/KravitzLab/Murrell2025/raw/refs/heads/main/Data/Bandit80.zip"
key_url = "https://github.com/KravitzLab/Murrell2025/raw/refs/heads/main/Data/Murrell2026_Key.csv"

zip_dir = "/content/Murrell2025_zipdata"
zip_path = os.path.join(zip_dir, "Bandit80.zip")
extract_root = os.path.join(zip_dir, "Bandit80_extracted")
key_path = os.path.join(zip_dir, "Murrell2026_Key.csv")

os.makedirs(zip_dir, exist_ok=True)

# download + unzip (fresh each run)
if os.path.exists(zip_path):
    os.remove(zip_path)
if os.path.isdir(extract_root):
    shutil.rmtree(extract_root)

print("Importing github.com/KravitzLab/Murrell2025/Data/Bandit80.zip ...")
urlretrieve(zip_url, zip_path)

os.makedirs(extract_root, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_root)

# if the zip contains a Bandit80 folder, use it; otherwise use extract_root
bandit_root = os.path.join(extract_root, "Bandit80")
local_parent_path = bandit_root if os.path.isdir(bandit_root) else extract_root

# load CSVs
feds, loaded_files, session_types = [], [], []

for dirpath, _, filenames in os.walk(local_parent_path):
    for file_name in sorted(filenames):
        if file_name.lower().endswith(".csv"):
            file_path = os.path.join(dirpath, file_name)
            strain_name = os.path.basename(os.path.dirname(file_path))

            df = fed3.load(file_path)
            df.name = file_name
            df["Strain"] = strain_name
            df["SourceFile"] = file_name

            feds.append(df)
            loaded_files.append(file_path)

            st = df["Session_Type"].dropna().astype(str)
            session_types.append(st.iloc[0] if len(st) else None)

print(f"Loaded {len(feds)} CSV files.")

# download + load key
urlretrieve(key_url, key_path)

key_df = pd.read_csv(key_path, encoding="utf-8-sig")
key_df["Mouse_ID"] = key_df["Mouse_ID"].astype(str).str.strip()

# match Mouse_ID by substring in filename base
def _base_lower(p):
    return os.path.splitext(os.path.basename(p))[0].lower()

files_df = pd.DataFrame({"filename": loaded_files, "Session_type": session_types})
files_df["_base"] = files_df["filename"].map(_base_lower)

mouse_ids = (
    key_df["Mouse_ID"]
    .dropna().astype(str).str.strip()
    .replace("", np.nan).dropna().unique().tolist()
)

rows = []
for fname, base in zip(files_df["filename"], files_df["_base"]):
    hits = [mid for mid in mouse_ids if mid.lower() in base]
    rows.append({"filename": fname, "Mouse_ID": hits[0] if len(hits) else None})

matched = pd.DataFrame(rows)

Key_Df = (
    files_df.drop(columns=["_base"])
    .merge(matched, on="filename", how="left")
    .merge(key_df.drop_duplicates("Mouse_ID"), on="Mouse_ID", how="left")
)

# display
grid = DataGrid(
    Key_Df.reset_index(drop=True),
    editable=True,
    selection_mode="cell",
    layout={"height": "420px"},
    base_row_size=28,
    base_column_size=120,
)
grid.default_renderer = TextRenderer(text_wrap=True)
display(grid)

In [ ]:
# @title Figure 3A, B, G

import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

# -----------------------------
# Panel A: 5 line win–stay sweep
# -----------------------------
win_stays = np.round(np.concatenate([[0.50, 0.55], np.arange(0.60, 1.01, 0.05)]), 2)
conditions = {
    "100:0": (1.0, 0.0),
    "90:10": (0.9, 0.1),
    "80:20": (0.8, 0.2),
    "70:30": (0.7, 0.3),
    "60:40": (0.6, 0.4),
}

rows = []
for label, (ph, pl) in conditions.items():
    seed = sum(ord(c) for c in label) % 1000
    for ws in win_stays:
        m, sem = simulate_mean_pHigh(ws, ph, pl, seed=seed)
        rows.append({
            "condition": label,
            "win_stay": ws,
            "mean_pHigh": m * 100,   # convert to %
            "sem_pHigh": sem * 100
        })

df_sweep = pd.DataFrame(rows)

# -----------------------------
# Load real datasets once
# -----------------------------
url_100 = (
    "https://raw.githubusercontent.com/KravitzLab/"
    "Murrell2026/refs/heads/main/Data/SummaryStats/Bandit100_metrics.csv"
)
real_100 = pd.read_csv(url_100)
real_100.columns = [c.strip() for c in real_100.columns]

url_8020 = (
    "https://raw.githubusercontent.com/KravitzLab/"
    "Murrell2026/refs/heads/main/Data/SummaryStats/Bandit80_metrics.csv"
)
real_8020 = pd.read_csv(url_8020)
real_8020.columns = [c.strip() for c in real_8020.columns]

sex_colors = {"M": "dodgerblue", "m": "dodgerblue", "F": "red", "f": "red"}

# -----------------------------
# Make 3 subplots (match look)
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ===== Panel A =====
ax = axes[0]

cmap = plt.cm.plasma
colors = cmap(np.linspace(0, 0.8, len(conditions)))
xfit = np.linspace(win_stays.min(), win_stays.max(), 300)

lines, labels = [], []
for color, (label, _) in zip(colors, conditions.items()):
    sub = df_sweep[df_sweep["condition"] == label].sort_values("win_stay")
    ax.scatter(sub["win_stay"], sub["mean_pHigh"], alpha=0.25, color=color)
    coef = np.polyfit(sub["win_stay"], sub["mean_pHigh"], deg=2)
    line, = ax.plot(xfit, np.polyval(coef, xfit), color=color, linewidth=2)
    lines.append(line)
    labels.append(label)

ax.set_xlabel("Win–stay probability")
ax.set_ylabel("Average accuracy (%)")
ax.set_ylim(45, 80)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

leg = ax.legend(
    lines, labels,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    handlelength=2.5,
    fontsize=12
)
for txt, line in zip(leg.get_texts(), lines):
    txt.set_color(line.get_color())

# ===== Panel B (100:0) =====
ax = axes[1]

ws_grid = np.round(np.linspace(0.5, 1.0, 25), 2)
sim_100 = np.array([
    simulate_mean_pHigh(ws, 1.0, 0.0, seed=1)[0] * 100
    for ws in ws_grid
])

sim_coef = np.polyfit(ws_grid, sim_100, deg=2)
ws_fit = np.linspace(ws_grid.min(), ws_grid.max(), 300)
sim_fit = np.polyval(sim_coef, ws_fit)

# Real data: Win-stay is stored as % in CSV -> convert to probability
x = pd.to_numeric(real_100["Win-stay"], errors="coerce") / 100.0
y = pd.to_numeric(real_100["TotalAccuracy"], errors="coerce")  # already %
mask = x.notna() & y.notna()
x = x[mask].to_numpy()
y = y[mask].to_numpy()

lin = linregress(x, y)
xfit_b = np.linspace(x.min(), x.max(), 300)
yfit_b = lin.slope * xfit_b + lin.intercept



r2 = lin.rvalue**2
pval = lin.pvalue
p_str = "<0.001" if pval < 0.001 else f"{pval:.3f}"

ax.plot(ws_fit, sim_fit, color="black", lw=2.5, zorder=1, label="Simulated 100:0 (quadratic)")

for sex, sub in real_100.groupby("Sex"):
    ax.scatter(
        pd.to_numeric(sub["Win-stay"], errors="coerce") / 100.0,
        pd.to_numeric(sub["TotalAccuracy"], errors="coerce"),
        color=sex_colors.get(sex, "gray"),
        alpha=0.5, s=50, zorder=3,
        label=f"Real {str(sex).upper()}"
    )

ax.plot(
    xfit_b, yfit_b, "--",
    color="gray", lw=2.5,
    label=f"Real fit (R²={r2:.2f}, p={p_str})"
)

ax.legend(loc="upper left", frameon=False, fontsize=12)
ax.set_xlabel("Win–stay probability")
ax.set_ylabel("Total accuracy (%)")
ax.set_ylim(40, 100)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

print(
    f"Real-data linear fit:\n"
    f"  TotalAccuracy = {lin.slope:.2f} × Win-stay + {lin.intercept:.2f}\n"
    f"  R² = {r2:.3f}, p = {pval:.3g}"
)

# ===== Panel C (80:20) =====
ax = axes[2]

sim_8020 = np.array([
    simulate_mean_pHigh(ws, 0.8, 0.2, seed=2)[0] * 100
    for ws in ws_grid
])

sim_coef = np.polyfit(ws_grid, sim_8020, deg=2)
ws_fit = np.linspace(ws_grid.min(), ws_grid.max(), 300)
sim_fit = np.polyval(sim_coef, ws_fit)

x = real_8020["Win-stay"].to_numpy(dtype=float)
y = real_8020["TotalAccuracy"].to_numpy(dtype=float)

lin = linregress(x, y)
xfit_c = np.linspace(x.min(), x.max(), 300)
yfit_c = lin.slope * xfit_c + lin.intercept

r2 = lin.rvalue**2
pval = lin.pvalue
p_str = "<0.001" if pval < 0.001 else f"{pval:.3f}"

ax.plot(ws_fit, sim_fit, color="black", lw=2.5, zorder=1, label="Simulated 80:20 (quadratic)")

for sex, sub in real_8020.groupby("Sex"):
    ax.scatter(
        sub["Win-stay"], sub["TotalAccuracy"],
        color=sex_colors.get(sex, "gray"),
        alpha=0.5, s=50, zorder=3,
        label=f"Real {str(sex).upper()}"
    )

ax.plot(
    xfit_c, yfit_c, "--",
    color="gray", lw=2.5,
    label=f"Real fit (R²={r2:.2f}, p={p_str})"
)

ax.legend(loc="upper left", frameon=False, fontsize=12)
ax.set_xlabel("Win–stay probability")
ax.set_ylabel("Total accuracy (%)")
ax.set_ylim(40, 100)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

print(
    f"80:20 real-data linear fit:\n"
    f"  TotalAccuracy = {lin.slope:.2f} × Win-stay + {lin.intercept:.2f}\n"
    f"  R² = {r2:.3f}, p = {pval:.3g}"
)

plt.tight_layout()
plt.show()

# -----------------------------
# Save PDF button
# -----------------------------
out = widgets.Output()
btn = widgets.Button(description="Save WSLS figure as PDF")

def _save_pdf(_):
    with out:
        clear_output()
        pdf_path = "Murrell_2026_Fig3_WSLS_3panel.pdf"
        fig.savefig(pdf_path, bbox_inches="tight")
        display(FileLink(pdf_path))

btn.on_click(_save_pdf)
display(btn, out)

In [ ]:
#@title Figure 3C
Image(url='https://github.com/KravitzLab/Murrell2026/blob/main/Figures/Murrell_2026jpgs/Murrell_2026_Figure3_schematic.jpg?raw=true', width=300)

In [ ]:
# @title Figure 3D Individual behaviour examples (F= red M= Blue)
# ----- Inputs -----
assert 'feds' in globals() and isinstance(feds, list) and len(feds) > 0, "No FED3 files loaded."
assert 'Key_Df' in globals() and isinstance(Key_Df, pd.DataFrame), "Build/rematch Key_Df first."

TIMESTAMP_COL_CANON = "MM:DD:YYYY hh:mm:ss"  # primary target column name

# ----- Helpers -----
def _find_time_col(df):
    # exact match first
    if TIMESTAMP_COL_CANON in df.columns:
        return TIMESTAMP_COL_CANON
    # tolerant search (case/space-insensitive)
    lc = {str(c).strip().lower(): c for c in df.columns}
    for key in lc:
        if key.replace(" ", "") in {"mm:dd:yyyyhh:mm:ss", "mm:dd:yyyy_hh:mm:ss", "mm/dd/yyyyhh:mm:ss"}:
            return lc[key]
    return None

def _parse_ts(series):
    # robust parsing; coerce errors to NaT
    return pd.to_datetime(series, errors="coerce", infer_datetime_format=True)


# ----- Plotting
files_list = feds

# metadata_df = copy of Key_Df
metadata_df = Key_Df.copy().reset_index(drop=True)
if 'filename' in metadata_df.columns:
    metadata_df['filename'] = metadata_df['filename'].astype(str).map(os.path.basename)

def _coerce_numeric_col(df, col, clip_upper=None, na_map=None):
    if col not in df.columns:
        return
    s = df[col]
    if na_map:
        s = s.replace(na_map)
    s = pd.to_numeric(s, errors='coerce')
    if clip_upper is not None:
        s.loc[s > clip_upper] = np.nan
    df[col] = s

def _plot_file_core(file_index):
    df = files_list[file_index].copy()
    full_name = getattr(df, 'name', f"File_{file_index}")
    file_basename = os.path.basename(str(full_name))

    # Preserve original index once
    if "Original_Timestamp" not in df.columns:
        df["Original_Timestamp"] = df.index

    # Attach metadata by filename (matching already done upstream)
    meta_row = None
    if 'filename' in metadata_df.columns:
        mr = metadata_df.loc[metadata_df['filename'] == file_basename]
        if not mr.empty:
            meta_row = mr.iloc[0]
    if meta_row is not None:
        for col in meta_row.index:
            if col == 'filename':
                continue
            if col not in df.columns:
                df[col] = meta_row[col]
            else:
                if pd.isna(df[col]).all() and pd.notna(meta_row[col]):
                    df[col] = meta_row[col]

    # Time + cleanup
    try:
        df['timestamp'] = pd.to_datetime(df.index)
    except Exception:
        df['timestamp'] = np.arange(len(df))

    _coerce_numeric_col(df, 'Poke_Time', clip_upper=2)
    _coerce_numeric_col(df, 'Retrieval_Time', na_map={"Timed_out": np.nan})

    if len(df) == 0:
        print(f"[!] Empty cropped dataframe for {file_basename}. Skipping plot.")
        return

    # Behavioral traces (needs f3b)
    true_left = f3b.true_probs(df, offset=5)[0]
    mouse_left = f3b.binned_paction(df, window=10)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(np.arange(len(true_left)), true_left, color="black", linewidth=2, alpha=0.5)

    color = "dodgerblue"
    if 'Sex' in df.columns and pd.notna(df['Sex']).any():
        try:
            color = "red" if str(df['Sex'].iloc[0]).strip().lower().startswith("f") else "dodgerblue"
        except Exception:
            pass
    ax.plot(np.arange(len(mouse_left)), mouse_left, color=color, linewidth=3, alpha=0.7)

    # ---- Clean look: remove ticks, labels, spines ----
    ax.set_xlabel("")                 # no x label
    ax.set_ylabel("")                 # no y label
    ax.tick_params(axis="both", which="both",
                   bottom=False, top=False, left=False, right=False,
                   labelbottom=False, labelleft=False)
    for s in ax.spines.values():      # remove axis lines
        s.set_visible(False)

    # ---- Textual y "labels" at y=1 and y=0 (not ticks) ----
    ax.text(-0.01, 1.0, "Right", transform=ax.get_yaxis_transform(),
            ha="right", va="center")
    ax.text(-0.01, 0.0, "Left",  transform=ax.get_yaxis_transform(),
            ha="right", va="center")

    # ---- Title-area arrow above the axes ----
    # spans full width; adjust y (1.08–1.15) if you need more/less space
    ax.annotate("",
                xy=(0.95, 1.05), xytext=(0.05, 1.05),
                xycoords="axes fraction",
                arrowprops=dict(arrowstyle="->", lw=5, color="0.6"))
    # Optional: small caption above the arrow (example: duration text)
    ax.text(0.4, 1.15, "2 days", transform=ax.transAxes, ha="left", va="bottom", color="0.5")
    sns.despine(left=True, bottom=True, top=True, right=True)
    plt.tight_layout()
    plt.show()

# ----- Simple UI: slider + status + output -----
N = len(files_list)
assert N > 0, "No files available after cropping."
idx_slider = widgets.IntSlider(min=0, max=max(0, N-1), step=1, value=0, description='File', continuous_update=True)
status_lbl = widgets.HTML()
out = widgets.Output()

def _status(idx):
    name = getattr(files_list[idx], 'name', f"File_{idx}")
    return f"Index: <b>{idx}</b> &nbsp;|&nbsp; File: <code>{os.path.basename(str(name))}</code> &nbsp;|&nbsp; Rows: {len(files_list[idx])}"

def _render(*_):
    idx = int(idx_slider.value)
    status_lbl.value = _status(idx)
    out.clear_output()
    with out:
        _plot_file_core(idx)

idx_slider.observe(_render, names='value')
display(widgets.VBox([idx_slider, status_lbl, out]))
_render()



In [ ]:
# @title Analyze Bandit metrics
from pathlib import Path
import os
def _find_time_col(df):
    for c in ["MM:DD:YYYY hh:mm:ss", "DateTime", "Datetime", "Timestamp", "timestamp", "datetime"]:
        if c in df.columns:
            return c
    return None

def _get_timestamp_series(df, ts_col="MM:DD:YYYY hh:mm:ss"):
    import pandas as pd
    if ts_col in df.columns:
        ts = pd.to_datetime(df[ts_col], format="%m:%d:%Y %H:%M:%S", errors="coerce")
        return pd.Series(ts, index=df.index)
    for cand in ["DateTime", "Datetime", "Timestamp", "timestamp", "datetime"]:
        if cand in df.columns:
            ts = pd.to_datetime(df[cand], errors="coerce")
            return pd.Series(ts, index=df.index)
    idx = df.index
    if isinstance(idx, pd.DatetimeIndex):
        return pd.Series(idx, index=df.index)
    return pd.to_datetime(pd.Series(idx, index=df.index), errors="coerce")

def _crop_last_24h(df):
    import pandas as pd
    dfc = df.copy()
    ts_col = _find_time_col(dfc)

    if ts_col is not None:
        ts_series = pd.to_datetime(dfc[ts_col], errors="coerce", infer_datetime_format=True)
        ts_series = pd.Series(ts_series, index=dfc.index)
    else:
        try:
            ts_idx = pd.to_datetime(dfc.index, errors="coerce", infer_datetime_format=True)
            ts_series = pd.Series(ts_idx, index=dfc.index)
        except Exception:
            return df  # no usable timestamps → return original

    if ts_series.isna().all():
        return df

    end = ts_series.max()
    if pd.isna(end):
        return df
    start = end - pd.Timedelta(hours=24)

    mask = ts_series.between(start, end, inclusive="both")
    cropped = dfc.loc[mask]

    if hasattr(df, "name"):
        cropped.name = df.name
    return cropped

def build_feds_cropped(sessions):
    """Return a list of sessions cropped to their last 24h."""
    return [_crop_last_24h(d) for d in sessions]

# Use it like this:
assert 'feds' in globals() and isinstance(feds, (list, tuple)) and len(feds) > 0, "No FED3 files available."
feds_cropped = build_feds_cropped(feds)

# Prefer cropped sessions downstream
_sessions = list(feds_cropped) if len(feds_cropped) > 0 else list(feds)
# ----- Build feds_cropped -----
feds_cropped = [_crop_last_24h(d) for d in feds]
# ---------- Inputs ----------
# Prefer cropped sessions
if 'feds_cropped' in globals() and isinstance(feds_cropped, (list, tuple)) and len(feds_cropped) > 0:
    _sessions = list(feds_cropped)
else:
    assert 'feds' in globals() and isinstance(feds, (list, tuple)) and len(feds) > 0, "No FED3 files available."
    _sessions = list(feds)

# metadata_df from Key_Df
if 'metadata_df' not in globals() or not isinstance(metadata_df, pd.DataFrame):
    assert 'Key_Df' in globals() and isinstance(Key_Df, pd.DataFrame), "Build/rematch Key_Df first."
    metadata_df = Key_Df.copy().reset_index(drop=True)

def _basename(pathlike) -> str:
    s = str(pathlike).replace("\\", "/")
    return s.split("/")[-1]

def _get_timestamp_series(df, ts_col="MM:DD:YYYY hh:mm:ss"):
    if ts_col in df.columns:
        ts = pd.to_datetime(df[ts_col], format="%m:%d:%Y %H:%M:%S", errors="coerce")
        return pd.Series(ts, index=df.index)
    for cand in ["DateTime", "Datetime", "Timestamp", "timestamp", "datetime"]:
        if cand in df.columns:
            ts = pd.to_datetime(df[cand], errors="coerce")
            return pd.Series(ts, index=df.index)
    idx = df.index
    if isinstance(idx, pd.DatetimeIndex):
        return pd.Series(idx, index=df.index)
    return pd.to_datetime(pd.Series(idx, index=df.index), errors="coerce")

def _split_day_night(df, ts_col="MM:DD:YYYY hh:mm:ss"):
    ts = _get_timestamp_series(df, ts_col=ts_col)
    valid = ts.notna()
    hrs = ts.dt.hour
    day_mask = valid & (hrs >= 7) & (hrs < 19)
    night_mask = valid & ~day_mask
    return df.loc[day_mask], df.loc[night_mask]

def compute_withinbout_lose_shift(c_df, max_gap_s=120):
    try:
        if "Event" not in c_df.columns or len(c_df) < 2:
            return np.nan
        events = c_df["Event"].to_numpy()
        times = _get_timestamp_series(c_df).to_numpy()
        total = shifted = 0
        for i in range(len(events) - 1):
            curr_evt, next_evt = events[i], events[i + 1]
            if curr_evt not in ("Left", "Right"):
                continue
            dt_s = (times[i + 1] - times[i]) / np.timedelta64(1, "s")
            if np.isnan(dt_s) or dt_s > max_gap_s:
                continue
            if next_evt == "Pellet":
                continue
            if next_evt in ("Left", "Right"):
                total += 1
                if next_evt != curr_evt:
                    shifted += 1
        return (shifted / total) if total > 0 else np.nan
    except Exception:
        return np.nan

def compute_withinbout_lose_stay(c_df, max_gap_s=120):
    try:
        if "Event" not in c_df.columns or len(c_df) < 2:
            return np.nan
        events = c_df["Event"].to_numpy()
        times = _get_timestamp_series(c_df).to_numpy()
        total = 0
        stayed = 0
        for i in range(len(events) - 1):
            curr_evt, next_evt = events[i], events[i + 1]
            if curr_evt not in ("Left", "Right"):
                continue
            dt_s = (times[i + 1] - times[i]) / np.timedelta64(1, "s")
            if np.isnan(dt_s) or dt_s > max_gap_s:
                continue
            if next_evt == "Pellet":
                continue
            if next_evt in ("Left", "Right"):
                total += 1
                if next_evt == curr_evt:
                    stayed += 1
        return (stayed / total) if total > 0 else np.nan
    except Exception:
        return np.nan

def compute_withinbout_win_stay(c_df, max_gap_s=120):
    try:
        if "Event" not in c_df.columns or len(c_df) < 3:
            return np.nan
        events = c_df["Event"].to_numpy()
        times = _get_timestamp_series(c_df).to_numpy()
        pellet_idx = [i for i in range(1, len(events) - 1) if events[i] == "Pellet"]
        total = same = 0
        for i in pellet_idx:
            prev_event, next_event = events[i - 1], events[i + 1]
            dt_s = (times[i + 1] - times[i]) / np.timedelta64(1, "s")
            if not np.isnan(dt_s) and dt_s <= max_gap_s:
                if prev_event in ("Left", "Right") and next_event in ("Left", "Right"):
                    total += 1
                    if next_event == prev_event:
                        same += 1
        return (same / total) if total > 0 else np.nan
    except Exception:
        return np.nan
def compute_withinbout_win_shift(c_df, max_gap_s=120):
    try:
        if "Event" not in c_df.columns or len(c_df) < 3:
            return np.nan
        events = c_df["Event"].to_numpy()
        times = _get_timestamp_series(c_df).to_numpy()
        pellet_idx = [i for i in range(1, len(events) - 1) if events[i] == "Pellet"]
        total = 0
        shifted = 0
        for i in pellet_idx:
            prev_event, next_event = events[i - 1], events[i + 1]
            dt_s = (times[i + 1] - times[i]) / np.timedelta64(1, "s")
            if np.isnan(dt_s) or dt_s > max_gap_s:
                continue
            if prev_event in ("Left", "Right") and next_event in ("Left", "Right"):
                total += 1
                if next_event != prev_event:
                    shifted += 1
        return (shifted / total) if total > 0 else np.nan
    except Exception:
        return np.nan

def compute_peak_accuracy(c_df):
    try:
        rev_avg = f3b.reversal_peh(c_df, (-10, 10), True)
        if len(rev_avg) == 0:
            return np.nan
        return float(np.mean(rev_avg[:10]))*100 if len(rev_avg) >= 10 else float(np.mean(rev_avg))*100
    except Exception:
        return np.nan

def estimate_daily_pellets(c_df):
    ts = _get_timestamp_series(c_df)
    valid_ts = ts.dropna()
    if valid_ts.size < 2:
        return np.nan
    duration_hours = (valid_ts.max() - valid_ts.min()).total_seconds() / 3600.0
    if duration_hours <= 0:
        return np.nan

    pellet_events = np.nan
    if "Pellet_Count" in c_df.columns and c_df["Pellet_Count"].notna().any():
        pc = pd.to_numeric(c_df["Pellet_Count"], errors="coerce")
        if pc.notna().any():
            diffs = pc.diff().fillna(0).clip(lower=0)
            pellet_events = float(diffs.sum())
            if pellet_events == 0 and pc.iloc[-1] >= pc.iloc[0]:
                pellet_events = float(pc.iloc[-1] - pc.iloc[0])
    if (pd.isna(pellet_events)) and ("Event" in c_df.columns):
        pellet_events = float((c_df["Event"] == "Pellet").sum())

    if pd.isna(pellet_events):
        return np.nan
    return (pellet_events / duration_hours) * 24.0

# ---------- Prepare metadata (merge once by filename) ----------
md = metadata_df.copy()
md['filename'] = md['filename'].astype(str).map(_basename)
if 'Mouse_ID' in md.columns:
    md['Mouse_ID'] = md['Mouse_ID'].astype(str).str.strip()
else:
    md['Mouse_ID'] = np.nan

# Keep only metadata columns we care about; rename to avoid accidental dupes
# (add/remove columns as needed)
meta_keep = [c for c in md.columns if c in {"filename", "Mouse_ID", "Session_type", "Cohort", "Strain", "Sex"}]
md_clean = md[meta_keep].drop_duplicates(subset=["filename"], keep="first")

# ---------- Compute metrics on the chosen sessions ----------
rows = []
for idx in tqdm(range(len(_sessions))):
    c_df = _sessions[idx]
    file_name = _basename(getattr(c_df, "name", f"File_{idx}"))

    try:
        clean_retrieval_time = pd.to_numeric(c_df.get("Retrieval_Time", pd.Series(dtype=float)), errors="coerce")
        clean_poke_time = pd.to_numeric(c_df.get("Poke_Time", pd.Series(dtype=float)), errors="coerce")
        clean_poke_time = clean_poke_time[clean_poke_time > 0]

        day_df, night_df = _split_day_night(c_df, ts_col="MM:DD:YYYY hh:mm:ss")

        row = {
            "filename": file_name,
            "PeakAccuracy": compute_peak_accuracy(c_df),
            "TotalAccuracy": f3b.accuracy(c_df)*100,  # total accuracy, not pre-reversal accuracy
            "Total_pellets": f3b.count_pellets(c_df),
            "Total_pokes": f3b.count_pokes(c_df),
            "PokesPerPellet": f3b.pokes_per_pellet(c_df),
            "RetrievalTime": clean_retrieval_time.median() if not clean_retrieval_time.empty else np.nan,
            "PokeTime": clean_poke_time.median() if not clean_poke_time.empty else np.nan,
            "Win-stay": compute_withinbout_win_stay(c_df),
            "Win-shift": compute_withinbout_win_shift(c_df),
            "Lose-shift": compute_withinbout_lose_shift(c_df),
            "Lose-stay": compute_withinbout_lose_stay(c_df),
            "daily pellets": estimate_daily_pellets(c_df),
            "PeakAccuracy_Day": compute_peak_accuracy(day_df),
            "PeakAccuracy_Night": compute_peak_accuracy(night_df),
            "Win-stay_Day": compute_withinbout_win_stay(day_df),
            "Win-stay_Night": compute_withinbout_win_stay(night_df),
            "Lose-shift_Day": compute_withinbout_lose_shift(day_df),
            "Lose-shift_Night": compute_withinbout_lose_shift(night_df),
        }
        rows.append(row)

    except Exception as e:
        print(f"Failed on {file_name} (idx {idx}): {e}")


Bandit80_metrics = pd.DataFrame(rows)
Bandit80_metrics = Bandit80_metrics.merge(md_clean, on="filename", how="left")
Bandit80_metrics = Bandit80_metrics.loc[:, ~Bandit80_metrics.columns.duplicated()]

csv_name = "Bandit80_metrics.csv"
Bandit80_metrics.to_csv(csv_name, index=False)

GROUP_COL = "Sex"
metrics = ["TotalAccuracy", "Win-stay", "Lose-stay", "daily pellets", "RetrievalTime", "Total_pokes"]
COLOR_MAP = {"F": "red", "M": "dodgerblue"}

def _norm_sex(x):
    s = str(x).strip().upper()
    if s in {"F", "FEMALE", "FEM"}: return "F"
    if s in {"M", "MALE"}: return "M"
    return "UNK"

def welch_p(a, b):
    a = pd.Series(a, dtype=float).dropna()
    b = pd.Series(b, dtype=float).dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return float(ttest_ind(a, b, equal_var=False, nan_policy="omit").pvalue)

# --- Preconditions ---
if "Bandit80_metrics" not in globals() or Bandit80_metrics is None or Bandit80_metrics.empty:
    raise RuntimeError("Bandit80_metrics is missing/empty. Run the metrics cell first.")

bm = Bandit80_metrics.copy()

# --- Ensure Sex (merge from metadata_df if needed) ---
if GROUP_COL not in bm.columns or bm[GROUP_COL].isna().all():
    if "metadata_df" not in globals() or metadata_df is None or metadata_df.empty:
        raise RuntimeError("Sex not found in Bandit80_metrics and metadata_df is missing/empty.")
    meta = metadata_df.copy()

    def _find_col(df, name):
        lc = {str(c).strip().lower(): c for c in df.columns}
        return lc.get(name.lower(), None)

    meta_sex = _find_col(meta, "Sex")
    if meta_sex is None:
        raise RuntimeError("metadata_df does not contain a 'Sex' column (case-insensitive).")

    meta_mouse = _find_col(meta, "Mouse_ID")
    bm_mouse   = _find_col(bm, "Mouse_ID")

    merged = None
    if meta_mouse is not None and bm_mouse is not None:
        meta_key = meta[[meta_mouse, meta_sex]].copy()
        meta_key.columns = ["Mouse_ID", "Sex"]
        meta_key["Mouse_ID"] = meta_key["Mouse_ID"].astype(str).str.strip()
        meta_key = meta_key.dropna(subset=["Mouse_ID"]).drop_duplicates("Mouse_ID")

        bm["Mouse_ID"] = bm[bm_mouse].astype(str).str.strip()
        merged = bm.merge(meta_key, on="Mouse_ID", how="left")

    if merged is None or merged["Sex"].isna().all():
        if "filename" not in bm.columns:
            if "File" in bm.columns:
                bm["filename"] = bm["File"].astype(str)
            else:
                raise RuntimeError("Need either Mouse_ID or filename/File in Bandit80_metrics to merge Sex.")
        if "filename" not in meta.columns:
            raise RuntimeError("Cannot fallback merge: metadata_df has no filename column.")

        bm["file_base"]   = bm["filename"].astype(str).apply(lambda p: os.path.basename(p))
        meta["file_base"] = meta["filename"].astype(str).apply(lambda p: os.path.basename(p))

        meta_key = meta[["file_base", meta_sex]].copy()
        meta_key.columns = ["file_base", "Sex"]
        meta_key = meta_key.dropna(subset=["file_base"]).drop_duplicates("file_base")

        merged = bm.merge(meta_key, on="file_base", how="left").drop(columns=["file_base"])

    bm = merged

#Create Long_Df for plotting
bm[GROUP_COL] = bm[GROUP_COL].apply(_norm_sex)
bm = bm[bm[GROUP_COL].isin(["F", "M"])].copy()
if bm.empty:
    raise RuntimeError("After merging/normalizing Sex, no rows with Sex in {F, M} were found.")

# --- Long format (your old way) ---
value_vars = [m for m in metrics if m in bm.columns]
if not value_vars:
    raise RuntimeError("None of the expected metric columns were found in Bandit100_metrics.")

id_vars = [c for c in ["filename", "Mouse_ID", "Strain", GROUP_COL] if c in bm.columns]
long_df = bm.melt(id_vars=id_vars, value_vars=value_vars, var_name="metric", value_name="value")

groups = [g for g in ["F", "M"] if g in long_df[GROUP_COL].unique()]
if len(groups) < 2:
    raise RuntimeError(f"Need both F and M present to compare; found: {groups}")

# --- Precompute raw p-values for ALL metrics (once) ---
raw_pvals = {}
for metric in metrics:
    if metric not in value_vars:
        raw_pvals[metric] = np.nan
        continue
    dfm = long_df[long_df["metric"] == metric].dropna(subset=["value"])
    a = dfm.loc[dfm[GROUP_COL] == groups[0], "value"]
    b = dfm.loc[dfm[GROUP_COL] == groups[1], "value"]
    raw_pvals[metric] = welch_p(a, b)


from google.colab import files
import ipywidgets as widgets
from IPython.display import display

def download_csv(b):
    files.download(csv_name)

download_button = widgets.Button(
    description="⬇️ Download summary stats (CSV)",
    button_style="primary",
)

download_button.on_click(download_csv)
display(download_button)

In [ ]:
#@title Figure 3E, F

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from statsmodels.stats.multitest import multipletests

def plot_metrics(
    long_df,
    metrics_subset,
    group_col="Sex",
    groups=("F", "M"),
    color_map=None,
    raw_pvals=None,
    bonferroni_metrics=None,
    ncols=3,
    figsize=None,
):
    df_plot = long_df[long_df["metric"].isin(metrics_subset)].copy()

    # ---- summary table for THIS subset only ----
    summary_long = (
        df_plot.dropna(subset=["value"])
              .groupby(["metric", group_col])["value"]
              .agg(n="count", mean="mean", sem="sem")
              .reset_index()
    )
    summary_table = summary_long.pivot(index="metric", columns=group_col, values=["n", "mean", "sem"])

    col_order = []
    for stat in ["n", "mean", "sem"]:
        for g in groups:
            if (stat, g) in summary_table.columns:
                col_order.append((stat, g))
    summary_table = summary_table[col_order]
    summary_table.columns = [f"{g}_{stat}" for stat, g in summary_table.columns]

    for g in groups:
        if f"{g}_mean" in summary_table.columns and f"{g}_sem" in summary_table.columns:
            m = summary_table[f"{g}_mean"]
            s = summary_table[f"{g}_sem"]
            summary_table[f"{g}_mean±SEM"] = np.where(
                m.notna(),
                m.map(lambda x: f"{x:.3f}") + " ± " + s.map(lambda x: f"{x:.3f}"),
                np.nan
            )

    keep_cols = []
    for g in groups:
        if f"{g}_n" in summary_table.columns:
            summary_table[f"{g}_n"] = summary_table[f"{g}_n"].astype("Int64")
            keep_cols.append(f"{g}_n")
        if f"{g}_mean±SEM" in summary_table.columns:
            keep_cols.append(f"{g}_mean±SEM")

    summary_table = summary_table.loc[metrics_subset, keep_cols]

    display(
        summary_table.style
            .format({c: "{:d}" for c in summary_table.columns if c.endswith("_n")})
            .set_caption("")
    )

    # ---- Bonferroni only for requested metrics ----
    if bonferroni_metrics:
        bonferroni_metrics = [m for m in bonferroni_metrics if m in metrics_subset]
        if bonferroni_metrics:
            bonf_raw = [raw_pvals.get(m, np.nan) for m in bonferroni_metrics]
            finite_mask = [np.isfinite(p) for p in bonf_raw]

            print("Bonferroni Corrected p-values (Female vs. Male comparison):\n")
            if any(finite_mask):
                bonf_names = [m for m, ok in zip(bonferroni_metrics, finite_mask) if ok]
                bonf_vals  = [p for p, ok in zip(bonf_raw, finite_mask) if ok]
                _, p_bonf, _, _ = multipletests(bonf_vals, alpha=0.05, method="bonferroni")
                bonf_map = dict(zip(bonf_names, p_bonf))

                for m in bonferroni_metrics:
                    p = bonf_map.get(m, np.nan)
                    print(f"- {m}: p = {p:.4f}" if np.isfinite(p) else f"- {m}: p = NA (Insufficient data)")
            else:
                for m in bonferroni_metrics:
                    print(f"- {m}: p = NA (Insufficient data)")
            print()

    # ---- plotting ----
    if color_map is None:
        color_map = {g: "0.7" for g in groups}
    pal = {g: color_map[g] for g in groups if g in color_map}

    n = len(metrics_subset)
    nrows = int(np.ceil(n / ncols))
    if figsize is None:
        figsize = (6, 3.0 * nrows)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for i, metric in enumerate(metrics_subset):
        ax = axes[i]
        dfm = df_plot[df_plot["metric"] == metric].dropna(subset=["value"])

        sns.boxplot(
            data=dfm, x=group_col, y="value",
            order=list(groups), palette=pal,
            ax=ax, linewidth=0,
            whiskerprops=dict(linewidth=0),
            capprops=dict(linewidth=0),
            medianprops=dict(color="0.4", linewidth=2),
        )
        for patch in ax.patches:
            patch.set_alpha(0.6)

        sns.stripplot(
            data=dfm, x=group_col, y="value",
            order=list(groups),
            color="white", edgecolor="black",
            linewidth=1, size=6,
            alpha=0.35, jitter=True, ax=ax
        )

        p_raw = raw_pvals.get(metric, np.nan) if raw_pvals is not None else np.nan
        label = "p<0.001" if np.isfinite(p_raw) and p_raw < 0.001 else (f"p={p_raw:.3f}" if np.isfinite(p_raw) else "p=NA")
        ax.set_title("")
        ax.set_xlabel("")
        ax.set_ylabel(metric, fontsize=12)
        ax.text(0.5, 1.02, label, transform=ax.transAxes, ha="center", va="bottom")
        sns.despine(ax=ax)

    for j in range(n, len(axes)):
        axes[j].set_axis_off()


_ = plot_metrics(
long_df=long_df,
metrics_subset=metrics[:2],
group_col=GROUP_COL,
groups=groups,
color_map=COLOR_MAP,
raw_pvals=raw_pvals,
bonferroni_metrics=metrics[:2],
ncols=3
)

# Below is additional information not shown in Figure 3

In [ ]:
#@title Lose-stay, Pellets, Total pokes and Pellet Retrieval time.
_ = plot_metrics(
    long_df=long_df,
    metrics_subset=metrics[2:],
    group_col=GROUP_COL,
    groups=groups,
    color_map=COLOR_MAP,
    raw_pvals=raw_pvals,
    bonferroni_metrics=[],
    ncols=3
)

In [ ]:
# @title Accuracy around switches

import os, re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Config ---
TRIALS = 11
COLOR_MAP = {"F": "red", "M": "dodgerblue"}

# --- Preconditions ---
if 'feds' not in globals() or not isinstance(feds, (list, tuple)) or len(feds) == 0:
    raise RuntimeError("No FED3 sessions found in `feds`.")
if 'metadata_df' not in globals() or metadata_df is None or metadata_df.empty:
    raise RuntimeError("metadata_df is missing/empty. Build it from the Key first.")

# --- Helpers ---
def _basename(x): return os.path.basename(str(x))

def _norm_sex(x):
    s = str(x).strip().upper()
    if s in {"F", "FEMALE", "FEM"}: return "F"
    if s in {"M", "MALE"}: return "M"
    return "UNK"

def _find_col(df, name):
    lc = {str(c).strip().lower(): c for c in df.columns}
    return lc.get(name.lower(), None)

def _extract_mouse_id_from_sess_name(name):
    """
    Expected patterns like:
      CDKL5_005_180_69_Bandit80_20251013.csv  -> CDKL5_005_180_69
      ChowHFD_670_Bandit80_20250728.csv       -> ChowHFD_670
    Returns the ID part (no extension).
    """
    base = _basename(name)
    base = re.sub(r"\.csv$", "", base, flags=re.IGNORECASE)

    # Strip trailing task/date: _Bandit80_YYYYMMDD, _FR1_YYYYMMDD, etc.
    base = re.sub(r"_(Bandit80|Bandit100|FR1|PR1)_\d{8}$", "", base, flags=re.IGNORECASE)
    return base

def _is_empty(x):
    if x is None: return True
    try:
        return len(x) == 0
    except Exception:
        try:
            return np.size(x) == 0
        except Exception:
            return True

# --- Build Sex lookup from metadata_df ---
meta = metadata_df.copy()
meta_sex = _find_col(meta, "Sex")
if meta_sex is None:
    raise RuntimeError("metadata_df does not contain a 'Sex' column (case-insensitive).")
meta_mouse = _find_col(meta, "Mouse_ID")

sex_lookup = {}

# Prefer Mouse_ID mapping if present
if meta_mouse is not None:
    tmp = meta[[meta_mouse, meta_sex]].copy()
    tmp.columns = ["Mouse_ID", "Sex"]
    tmp["Mouse_ID"] = tmp["Mouse_ID"].astype(str).str.strip()
    tmp["Sex"] = tmp["Sex"].apply(_norm_sex)
    tmp = tmp.dropna(subset=["Mouse_ID"]).drop_duplicates("Mouse_ID")
    sex_lookup = dict(zip(tmp["Mouse_ID"], tmp["Sex"]))

# Fallback: if metadata_df has filename, build filename->sex map too
file_sex_lookup = {}
if "filename" in meta.columns:
    tmp2 = meta[["filename", meta_sex]].copy()
    tmp2["file_base"] = tmp2["filename"].astype(str).apply(_basename)
    tmp2["Sex"] = tmp2[meta_sex].apply(_norm_sex)
    tmp2 = tmp2.dropna(subset=["file_base"]).drop_duplicates("file_base")
    file_sex_lookup = dict(zip(tmp2["file_base"], tmp2["Sex"]))

# --- Build rev_df ---
rows = []
for i, sess in enumerate(feds_cropped):
    sess_name = getattr(sess, "name", f"session_{i}")
    base = _basename(sess_name)

    # Get an ID from session filename and look up Sex
    mouse_id = _extract_mouse_id_from_sess_name(base)
    sex = sex_lookup.get(mouse_id, "UNK")

    # fallback: try direct filename basename lookup (only if metadata has filename)
    if sex == "UNK" and file_sex_lookup:
        sex = file_sex_lookup.get(base, "UNK")

    # compute peri-switch trials using your helper
    try:
        peh = f3b.reversal_peh(sess, (-TRIALS, TRIALS), return_avg=False)
    except Exception as e:
        print(f"[skip] {base}: reversal_peh failed: {e}")
        continue

    if _is_empty(peh) or sex not in {"F", "M"}:
        continue

    for tr in list(peh):
        arr = np.asarray(tr).ravel()
        for t, v in enumerate(arr):
            rows.append({
                "Timepoint": t - TRIALS + 1,
                "Value": float(v) if np.isfinite(v) else np.nan,
                "Sex": sex
            })

rev_df = pd.DataFrame(rows)
rev_df = rev_df[np.isfinite(rev_df["Value"])]
rev_df = rev_df[rev_df["Timepoint"] != 0]  # optional
if rev_df.empty:
    raise RuntimeError("No peri-switch data produced (after filtering to F/M).")

# --- Plot ---
group_order = [g for g in ["F", "M"] if g in rev_df["Sex"].unique()]
palette = {g: COLOR_MAP[g] for g in group_order}

plt.figure(figsize=(5, 4))
ax = sns.lineplot(
    data=rev_df.sort_values(["Sex", "Timepoint"]),
    x="Timepoint",
    y="Value",
    hue="Sex",
    hue_order=group_order,
    palette=palette,
    estimator="mean",
    errorbar="se",
    n_boot=0,
    lw=2
)

ax.axvline(x=0, color="darkgrey", linestyle="--", linewidth=1.25)
ymin, ymax = ax.get_ylim()
ax.text(0.5, ymin + 0.95*(ymax - ymin), "Switch", color="darkgrey",
        fontsize=12, ha="left", va="top")

ax.set_xlabel("Trials from switch")
ax.set_ylabel("Accuracy (%)")
ax.set_title("")
ax.legend(title="", frameon=False)
sns.despine()
plt.tight_layout()
plt.show()